In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
IMG_SIZE = 256

IMAGE_DIR = "/content/drive/MyDrive/glaucoma detection/Images_refuge"
MASK_DIR  = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"



In [ ]:
import os

image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".png"))
])

mask_files = sorted([
    f for f in os.listdir(MASK_DIR)
    if f.lower().endswith((".png", ".jpg"))
])

print("Images:", image_files[:5])
print("Masks :", mask_files[:5])
print("Total images:", len(image_files))
print("Total masks :", len(mask_files))

assert len(image_files) == len(mask_files), "❌ Image-mask count mismatch"


In [ ]:
images = []
disc_masks = []
cup_masks = []

for img_name, mask_name in tqdm(zip(image_files, mask_files), total=len(image_files)):

    img_path = os.path.join(IMAGE_DIR, img_name)
    mask_path = os.path.join(MASK_DIR, mask_name)

    image = cv2.imread(img_path)
    mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if image is None or mask is None:
        continue

    # Resize
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    mask  = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

    # Normalize image
    image = image / 255.0

    # Create disc & cup masks
    disc = (mask == 1).astype(np.float32)
    cup  = (mask == 2).astype(np.float32)

    images.append(image)
    disc_masks.append(disc)
    cup_masks.append(cup)


In [ ]:
X = np.array(images, dtype=np.float32)
Y_disc = np.array(disc_masks, dtype=np.float32)
Y_cup  = np.array(cup_masks, dtype=np.float32)

print("X shape      :", X.shape)
print("Disc mask    :", Y_disc.shape)
print("Cup mask     :", Y_cup.shape)


In [ ]:
Y_disc = np.expand_dims(Y_disc, axis=-1)
Y_cup  = np.expand_dims(Y_cup, axis=-1)


In [ ]:
X_train, X_val, Yd_train, Yd_val, Yc_train, Yc_val = train_test_split(
    X, Y_disc, Y_cup,
    test_size=0.2,
    random_state=42
)


In [ ]:
print("Train images :", X_train.shape)
print("Val images   :", X_val.shape)
print("Train disc   :", Yd_train.shape)
print("Train cup    :", Yc_train.shape)


In [ ]:
import matplotlib.pyplot as plt

i = np.random.randint(0, len(X_train))

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(X_train[i])
plt.title("Fundus Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(Yd_train[i].squeeze(), cmap="gray")
plt.title("Disc Mask")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(Yc_train[i].squeeze(), cmap="gray")
plt.title("Cup Mask")
plt.axis("off")

plt.show()


In [ ]:
import os

IMAGE_DIR = "/content/drive/MyDrive/glaucoma detection/Images_refuge"
MASK_DIR  = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"

image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")])
mask_files  = sorted([f for f in os.listdir(MASK_DIR) if f.endswith(".png")])

print("Total images:", len(image_files))
print("Total masks :", len(mask_files))

assert len(image_files) == len(mask_files), "❌ Image–mask count mismatch"

print("✅ Image-mask count matches")


In [ ]:
import os
import cv2
import numpy as np

MASK_DIR = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"

disc_masks = sorted([f for f in os.listdir(MASK_DIR) if "_disc" in f])
cup_masks  = sorted([f for f in os.listdir(MASK_DIR) if "_cup" in f])

print("Disc masks:", len(disc_masks))
print("Cup masks :", len(cup_masks))

assert len(disc_masks) == len(cup_masks), "❌ Disc–cup count mismatch"


In [ ]:
empty_disc = 0
empty_cup = 0

for d, c in zip(disc_masks, cup_masks):
    disc = cv2.imread(os.path.join(MASK_DIR, d), 0)
    cup  = cv2.imread(os.path.join(MASK_DIR, c), 0)

    if np.sum(disc > 0) < 100:
        empty_disc += 1

    if np.sum(cup > 0) < 50:
        empty_cup += 1

print("Empty disc masks:", empty_disc)
print("Empty cup masks :", empty_cup)


In [ ]:
for i in range(5):  # random few samples
    img = cv2.imread(os.path.join(IMAGE_DIR, image_files[i]))
    mask = cv2.imread(os.path.join(MASK_DIR, mask_files[i]), 0)

    assert img.shape[:2] == mask.shape[:2], \
        f"❌ Shape mismatch at {image_files[i]}"

print("✅ Image-mask spatial alignment OK")


In [ ]:
import os
import cv2
import numpy as np

MASK_DIR = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"

disc_masks = sorted([f for f in os.listdir(MASK_DIR) if "_disc" in f])
cup_masks  = sorted([f for f in os.listdir(MASK_DIR) if "_cup" in f])

assert len(disc_masks) == len(cup_masks), "Disc–cup count mismatch"


In [ ]:
disc_areas = []
cup_areas = []

for d, c in zip(disc_masks, cup_masks):
    disc = cv2.imread(os.path.join(MASK_DIR, d), 0)
    cup  = cv2.imread(os.path.join(MASK_DIR, c), 0)

    disc_areas.append(np.sum(disc > 0))
    cup_areas.append(np.sum(cup > 0))


In [ ]:
import numpy as np
import cv2
import os

disc_areas = []
cup_areas = []

for fname in mask_files:
    mask_path = os.path.join(MASK_DIR, fname)
    mask = cv2.imread(mask_path, 0)

    if mask is None:
        continue

    disc_area = np.sum(mask == 1)  # DISC
    cup_area  = np.sum(mask == 2)  # CUP

    # Only keep valid masks
    if disc_area > 0 and cup_area > 0:
        disc_areas.append(disc_area)
        cup_areas.append(cup_area)

disc_areas = np.array(disc_areas)
cup_areas  = np.array(cup_areas)

print("Disc area stats:")
print(" Min:", disc_areas.min())
print(" Max:", disc_areas.max())
print(" Mean:", disc_areas.mean())

print("\nCup area stats:")
print(" Min:", cup_areas.min())
print(" Max:", cup_areas.max())
print(" Mean:", cup_areas.mean())


In [ ]:
clean_images = []
clean_masks = []

bad_count = 0

for img_name, mask_name in zip(image_files, mask_files):
    mask_path = os.path.join(MASK_DIR, mask_name)
    mask = cv2.imread(mask_path, 0)

    if mask is None:
        bad_count += 1
        continue

    disc_area = np.sum(mask == 1)
    cup_area  = np.sum(mask == 2)

    # strict quality rules
    if disc_area == 0:
        bad_count += 1
        continue
    if cup_area == 0:
        bad_count += 1
        continue
    if cup_area >= disc_area:
        bad_count += 1
        continue

    clean_images.append(img_name)
    clean_masks.append(mask_name)

print("✅ Clean samples:", len(clean_images))
print("❌ Removed samples:", bad_count)


In [ ]:
import cv2
import numpy as np

IMG_SIZE = 256

def preprocess_fundus(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # resize
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # extract green channel
    green = img[:, :, 1]

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    green = clahe.apply(green)

    # normalize to [0,1]
    green = green / 255.0

    return green


In [ ]:
def preprocess_mask(mask_path):
    mask = cv2.imread(mask_path, 0)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

    disc = (mask == 1).astype(np.float32)
    cup  = (mask == 2).astype(np.float32)

    return disc, cup


In [ ]:
X = []
Y_disc = []
Y_cup = []

for img_name, mask_name in zip(clean_images, clean_masks):
    img_path  = os.path.join(IMAGE_DIR, img_name)
    mask_path = os.path.join(MASK_DIR, mask_name)

    X.append(preprocess_fundus(img_path))
    disc, cup = preprocess_mask(mask_path)
    Y_disc.append(disc)
    Y_cup.append(cup)

X = np.array(X)[..., np.newaxis]
Y_disc = np.array(Y_disc)[..., np.newaxis]
Y_cup  = np.array(Y_cup)[..., np.newaxis]

print("X:", X.shape)
print("Disc:", Y_disc.shape)
print("Cup:", Y_cup.shape)


In [ ]:
import matplotlib.pyplot as plt

i = np.random.randint(len(X))

plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(X[i].squeeze(), cmap="gray")
plt.title("Fundus")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(Y_disc[i].squeeze(), cmap="gray")
plt.title("Disc")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(Y_cup[i].squeeze(), cmap="gray")
plt.title("Cup")
plt.axis("off")

plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, \
Yd_train, Yd_val, \
Yc_train, Yc_val = train_test_split(
    X, Y_disc, Y_cup,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)


In [ ]:
import tensorflow as tf

def dice_loss(y_true, y_pred):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return 1 - (2. * intersection + smooth) / \
           (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, UpSampling2D,
    concatenate, Activation, BatchNormalization, Multiply
)
from tensorflow.keras.models import Model


In [ ]:
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


In [ ]:
def attention_gate(x, g, filters):
    theta_x = Conv2D(filters, 1, padding="same")(x)
    phi_g   = Conv2D(filters, 1, padding="same")(g)

    add = Activation("relu")(theta_x + phi_g)
    psi = Conv2D(1, 1, padding="same")(add)
    psi = Activation("sigmoid")(psi)

    return Multiply()([x, psi])


In [ ]:
def Attention_UNet(input_shape=(256,256,1)):
    inputs = Input(input_shape)

    # Encoder
    c1 = conv_block(inputs, 32)
    p1 = MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = MaxPooling2D()(c3)

    c4 = conv_block(p3, 256)

    # Decoder
    u3 = UpSampling2D()(c4)
    a3 = attention_gate(c3, u3, 128)
    u3 = concatenate([u3, a3])
    c5 = conv_block(u3, 128)

    u2 = UpSampling2D()(c5)
    a2 = attention_gate(c2, u2, 64)
    u2 = concatenate([u2, a2])
    c6 = conv_block(u2, 64)

    u1 = UpSampling2D()(c6)
    a1 = attention_gate(c1, u1, 32)
    u1 = concatenate([u1, a1])
    c7 = conv_block(u1, 32)

    # Outputs
    disc_output = Conv2D(1, 1, activation="sigmoid", name="disc_output")(c7)
    cup_output  = Conv2D(1, 1, activation="sigmoid", name="cup_output")(c7)

    return Model(inputs, [disc_output, cup_output])


In [ ]:
model = Attention_UNet(input_shape=(256,256,1))
model.summary()


In [ ]:
def dice_loss(y_true, y_pred):
    smooth = 1e-6
    intersection = tf.reduce_sum(y_true * y_pred)
    return 1 - (2.*intersection + smooth) / \
           (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss={
        "disc_output": bce_dice_loss,
        "cup_output": bce_dice_loss
    },
    metrics={
        "disc_output": ["accuracy"],
        "cup_output": ["accuracy"]
    }
)


In [ ]:
history = model.fit(
    X_train,
    {"disc_output": Yd_train, "cup_output": Yc_train},
    validation_data=(X_val,
        {"disc_output": Yd_val, "cup_output": Yc_val}),
    epochs=40,
    batch_size=4
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

idx = np.random.randint(0, len(X_val))

disc_pred, cup_pred = model.predict(X_val[idx:idx+1])

disc_bin = (disc_pred[0,:,:,0] > 0.5).astype(np.uint8)
cup_bin  = (cup_pred[0,:,:,0] > 0.5).astype(np.uint8)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(X_val[idx].squeeze(), cmap="gray")
plt.title("Fundus")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(disc_bin, cmap="gray")
plt.title("Predicted Disc")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(cup_bin, cmap="gray")
plt.title("Predicted Cup")
plt.axis("off")

plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, \
Y_disc_train, Y_disc_val, \
Y_cup_train, Y_cup_val = train_test_split(
    X, Y_disc, Y_cup,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape, Y_disc_train.shape, Y_cup_train.shape)
print("Val  :", X_val.shape, Y_disc_val.shape, Y_cup_val.shape)


In [ ]:
import numpy as np

def dice_score(y_true, y_pred, smooth=1e-6):
    y_true = y_true.flatten()
    y_pred = (y_pred.flatten() > 0.5).astype(np.float32)
    intersection = np.sum(y_true * y_pred)
    return (2 * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)

disc_dice = []
cup_dice = []

for i in range(len(X_val)):
    pred_disc, pred_cup = model.predict(X_val[i:i+1], verbose=0)

    disc_dice.append(
        dice_score(Y_disc_val[i,:,:,0], pred_disc[0,:,:,0])
    )
    cup_dice.append(
        dice_score(Y_cup_val[i,:,:,0], pred_cup[0,:,:,0])
    )

print("✅ Mean Disc Dice:", np.mean(disc_dice))
print("✅ Mean Cup Dice :", np.mean(cup_dice))


In [ ]:
import cv2
import numpy as np
import pandas as pd

records = []

for i in range(len(X_val)):
    pred_disc, pred_cup = model.predict(X_val[i:i+1], verbose=0)

    disc_bin = (pred_disc[0,:,:,0] > 0.5).astype(np.uint8)
    cup_bin  = (pred_cup[0,:,:,0] > 0.5).astype(np.uint8)

    # Areas
    disc_area = np.sum(disc_bin)
    cup_area  = np.sum(cup_bin)

    if disc_area == 0 or cup_area == 0:
        continue

    # Vertical diameters
    disc_ys = np.where(disc_bin > 0)[0]
    cup_ys  = np.where(cup_bin > 0)[0]

    disc_vd = disc_ys.max() - disc_ys.min()
    cup_vd  = cup_ys.max() - cup_ys.min()

    vertical_cdr = cup_vd / (disc_vd + 1e-6)
    area_cdr = cup_area / (disc_area + 1e-6)

    records.append([
        i, disc_vd, cup_vd,
        disc_area, cup_area,
        vertical_cdr, area_cdr
    ])

df_cdr = pd.DataFrame(
    records,
    columns=[
        "index",
        "disc_vertical_diameter",
        "cup_vertical_diameter",
        "disc_area",
        "cup_area",
        "vertical_cdr",
        "area_cdr"
    ]
)

df_cdr.to_csv("refuge_val_cdr_features.csv", index=False)
df_cdr.head()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/glaucoma detection/refuge_val_cdr_features.csv"

df_cdr.to_csv(SAVE_PATH, index=False)

print("✅ File saved to:", SAVE_PATH)


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/glaucoma detection/refuge_val_cdr_features.csv"
)

print(df.head())


In [ ]:
df["image_name"] = df.index.map(lambda x: f"val_{x}.jpg")


In [ ]:
if "index" in df.columns:
    df = df.drop(columns=["index"])


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/glaucoma detection/refuge_val_cdr_features.csv"

df.to_csv(SAVE_PATH, index=False)

print("✅ CSV updated and saved successfully")


In [ ]:
df.columns


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/glaucoma detection/attention_unet_refuge.keras"

model.save(SAVE_PATH)

print("✅ Attention U-Net saved at:", SAVE_PATH)


In [ ]:
model.save_weights("/content/drive/MyDrive/glaucoma detection/attention_unet_refuge.weights.h5")


In [ ]:
from tensorflow.keras.models import load_model

MODEL_PATH = "/content/drive/MyDrive/glaucoma detection/attention_unet_refuge.keras"

model = load_model(MODEL_PATH, compile=False)
print("✅ Attention U-Net loaded")


In [ ]:
import os
import cv2
import numpy as np

IMG_DIR  = "/content/drive/MyDrive/glaucoma detection/Images_refuge"
MASK_DIR = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"

IMG_SIZE = 256

image_files = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".jpg")])
mask_files  = sorted([f for f in os.listdir(MASK_DIR) if f.endswith(".png")])

assert len(image_files) == len(mask_files), "❌ Image-mask mismatch"
print(f"✅ Total samples: {len(image_files)}")


In [ ]:
def dice_coef(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection) / (np.sum(y_true) + np.sum(y_pred) + 1e-7)

def iou_score(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return intersection / (union + 1e-7)

def pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def sensitivity(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP / (TP + FN + 1e-7)

def specificity(y_true, y_pred):
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    return TN / (TN + FP + 1e-7)


In [ ]:
disc_metrics = []
cup_metrics  = []

for img_name, mask_name in zip(image_files, mask_files):

    # Load image
    img = cv2.imread(os.path.join(IMG_DIR, img_name))
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    # Load mask
    mask = cv2.imread(os.path.join(MASK_DIR, mask_name), 0)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))

    disc_gt = (mask == 255).astype(np.uint8)
    cup_gt  = (mask == 128).astype(np.uint8)

    # Predict
    pred_disc, pred_cup = model.predict(img, verbose=0)

    disc_pred = (pred_disc[0, :, :, 0] > 0.5).astype(np.uint8)
    cup_pred  = (pred_cup[0, :, :, 0] > 0.5).astype(np.uint8)

    # Metrics
    disc_metrics.append([
        dice_coef(disc_gt, disc_pred),
        iou_score(disc_gt, disc_pred),
        pixel_accuracy(disc_gt, disc_pred),
        sensitivity(disc_gt, disc_pred),
        specificity(disc_gt, disc_pred)
    ])

    cup_metrics.append([
        dice_coef(cup_gt, cup_pred),
        iou_score(cup_gt, cup_pred),
        pixel_accuracy(cup_gt, cup_pred),
        sensitivity(cup_gt, cup_pred),
        specificity(cup_gt, cup_pred)
    ])


In [ ]:
# Load image (GRAYSCALE — IMPORTANT)
img = cv2.imread(os.path.join(IMG_DIR, img_name), cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img = img / 255.0

# Add channel + batch dimensions
img = np.expand_dims(img, axis=-1)   # (256, 256, 1)
img = np.expand_dims(img, axis=0)    # (1, 256, 256, 1)


In [ ]:
pred_disc, pred_cup = model.predict(img, verbose=0)


In [ ]:
model.input_shape


In [ ]:
print(model.output)
print(model.output_shape)


In [ ]:
import cv2
import numpy as np

IMG_SIZE = 256

def preprocess_single_image(img_path):
    img = cv2.imread(img_path)

    # Convert to grayscale
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Resize
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Normalize
    img = img.astype(np.float32) / 255.0

    # Add channel + batch dimensions
    img = np.expand_dims(img, axis=-1)  # (256,256,1)
    img = np.expand_dims(img, axis=0)   # (1,256,256,1)

    return img


In [ ]:
from tensorflow.keras.models import load_model

MODEL_PATH = "/content/drive/MyDrive/glaucoma detection/attention_unet_refuge.keras"

model = load_model(MODEL_PATH, compile=False)
print("✅ Attention U-Net loaded")

print("Model output shape:", model.output_shape)


In [ ]:
IMAGE_DIR = "/content/drive/MyDrive/glaucoma detection/Images_refuge"
IMAGE_NAME = sorted(os.listdir(IMAGE_DIR))[0]   # first image
IMAGE_PATH = os.path.join(IMAGE_DIR, IMAGE_NAME)

print("Using image:", IMAGE_PATH)


In [ ]:
IMG_SIZE = 256

def preprocess_single_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)   # 🔴 FIX
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)  # (256,256,1)
    img = np.expand_dims(img, axis=0)   # (1,256,256,1)
    return img


In [ ]:
img = preprocess_single_image(IMAGE_PATH)

pred_disc, pred_cup = model.predict(img, verbose=0)

# Remove batch & channel dims
pred_disc = pred_disc[0, :, :, 0]
pred_cup  = pred_cup[0, :, :, 0]

# Binarize
disc_bin = (pred_disc > 0.5).astype(np.uint8)
cup_bin  = (pred_cup > 0.5).astype(np.uint8)


In [ ]:
MASK_DIR = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"

mask_path = os.path.join(MASK_DIR, IMAGE_NAME.replace(".jpg", ".png"))
mask = cv2.imread(mask_path, 0)

gt_disc = (mask == 255).astype(np.uint8)
gt_cup  = (mask == 128).astype(np.uint8)


In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img[0,:,:,0], cmap="gray")
plt.title("Fundus")

plt.subplot(1,3,2)
plt.imshow(disc_bin, cmap="gray")
plt.title("Predicted Disc")

plt.subplot(1,3,3)
plt.imshow(cup_bin, cmap="gray")
plt.title("Predicted Cup")

plt.axis("off")
plt.show()


In [ ]:
def dice_coef(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    return (2. * inter) / (np.sum(y_true) + np.sum(y_pred) + 1e-6)

def iou_score(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - inter
    return inter / (union + 1e-6)

def pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def sensitivity(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn + 1e-6)

def specificity(y_true, y_pred):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tn / (tn + fp + 1e-6)


In [ ]:
# Resize GT mask to model size
mask_resized = cv2.resize(
    mask,
    (256, 256),
    interpolation=cv2.INTER_NEAREST
)

gt_disc = (mask_resized == 255).astype(np.uint8)
gt_cup  = (mask_resized == 128).astype(np.uint8)

print("GT disc shape:", gt_disc.shape)
print("Pred disc shape:", disc_bin.shape)


In [ ]:
def dice_coef(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    return (2. * inter) / (np.sum(y_true) + np.sum(y_pred) + 1e-6)

def iou_score(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - inter
    return inter / (union + 1e-6)

def pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def sensitivity(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn + 1e-6)

def specificity(y_true, y_pred):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tn / (tn + fp + 1e-6)


In [ ]:
metrics = {
    "disc_dice": dice_coef(gt_disc, disc_bin),
    "disc_iou": iou_score(gt_disc, disc_bin),
    "disc_pixel_acc": pixel_accuracy(gt_disc, disc_bin),
    "disc_sensitivity": sensitivity(gt_disc, disc_bin),
    "disc_specificity": specificity(gt_disc, disc_bin),

    "cup_dice": dice_coef(gt_cup, cup_bin),
    "cup_iou": iou_score(gt_cup, cup_bin),
    "cup_pixel_acc": pixel_accuracy(gt_cup, cup_bin),
    "cup_sensitivity": sensitivity(gt_cup, cup_bin),
    "cup_specificity": specificity(gt_cup, cup_bin),
}

df_metrics = pd.DataFrame(metrics, index=[IMAGE_NAME])
df_metrics


Evaluation metrices

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/glaucoma detection/attention_unet_refuge.keras"
model = load_model(MODEL_PATH, compile=False)

print("✅ Attention U-Net loaded")
print("Model outputs:", model.output_shape)


In [ ]:
def dice_coef(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    return (2. * inter) / (np.sum(y_true) + np.sum(y_pred) + 1e-6)

def iou_score(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - inter
    return inter / (union + 1e-6)

def pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def sensitivity(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn + 1e-6)

def specificity(y_true, y_pred):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tn / (tn + fp + 1e-6)


In [ ]:
IMAGE_DIR = "/content/drive/MyDrive/glaucoma detection/Images_refuge"
MASK_DIR  = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"

image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")])


In [ ]:
# --- Load image ---
img = cv2.imread(os.path.join(IMAGE_DIR, fname), cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (256, 256))
img = img / 255.0

# add channel + batch dimension
img = np.expand_dims(img, axis=-1)  # (256,256,1)
img = np.expand_dims(img, axis=0)   # (1,256,256,1)


In [ ]:
# Predict
pred_disc, pred_cup = model.predict(img, verbose=0)

pred_disc = (pred_disc[0, :, :, 0] > 0.5).astype(np.uint8)
pred_cup  = (pred_cup[0, :, :, 0] > 0.5).astype(np.uint8)


In [ ]:
print("Model input shape:", model.input_shape)


In [ ]:
import cv2
import numpy as np
import os

def load_grayscale_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image {img_path}")
    img = cv2.resize(img, (256, 256))
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)  # (256,256,1)
    img = np.expand_dims(img, axis=0)   # (1,256,256,1)
    return img


In [ ]:
# example image
IMAGE_DIR = "/content/drive/MyDrive/glaucoma detection/Images_refuge"
fname = sorted(os.listdir(IMAGE_DIR))[0]

img_path = os.path.join(IMAGE_DIR, fname)
img = load_grayscale_image(img_path)

pred_disc, pred_cup = model.predict(img, verbose=0)

pred_disc = (pred_disc[0, :, :, 0] > 0.5).astype(np.uint8)
pred_cup  = (pred_cup[0, :, :, 0] > 0.5).astype(np.uint8)


In [ ]:
print("Input shape:", img.shape)
print("Model input:", model.input_shape)


In [ ]:
import numpy as np

def dice_coef(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    return (2. * inter) / (np.sum(y_true) + np.sum(y_pred) + 1e-6)

def iou_score(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - inter
    return inter / (union + 1e-6)

def pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def sensitivity(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn + 1e-6)

def specificity(y_true, y_pred):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tn / (tn + fp + 1e-6)


In [ ]:
import cv2

def load_mask(mask_path, value):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (256, 256))
    return (mask == value).astype(np.uint8)


In [ ]:
import pandas as pd
import os

results = []

image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")])

for fname in image_files:
    img_path  = os.path.join(IMAGE_DIR, fname)
    mask_path = os.path.join(MASK_DIR, fname.replace(".jpg", ".png"))

    # Load image
    img = load_grayscale_image(img_path)

    # Ground truth masks
    gt_disc = load_mask(mask_path, 255)
    gt_cup  = load_mask(mask_path, 128)

    # Predict
    pred_disc, pred_cup = model.predict(img, verbose=0)
    pred_disc = (pred_disc[0, :, :, 0] > 0.5).astype(np.uint8)
    pred_cup  = (pred_cup[0, :, :, 0] > 0.5).astype(np.uint8)

    results.append({
        "image": fname,

        "disc_dice": dice_coef(gt_disc, pred_disc),
        "disc_iou": iou_score(gt_disc, pred_disc),
        "disc_pixel_acc": pixel_accuracy(gt_disc, pred_disc),
        "disc_sensitivity": sensitivity(gt_disc, pred_disc),
        "disc_specificity": specificity(gt_disc, pred_disc),

        "cup_dice": dice_coef(gt_cup, pred_cup),
        "cup_iou": iou_score(gt_cup, pred_cup),
        "cup_pixel_acc": pixel_accuracy(gt_cup, pred_cup),
        "cup_sensitivity": sensitivity(gt_cup, pred_cup),
        "cup_specificity": specificity(gt_cup, pred_cup),
    })


In [ ]:
IMAGE_DIR = "/content/drive/MyDrive/glaucoma detection/Images_refuge"
MASK_DIR  = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"

OUT_CSV = "/content/drive/MyDrive/glaucoma detection/segmentation_metrics_refuge.csv"


In [ ]:
df_metrics = pd.DataFrame(results)
df_metrics.to_csv(OUT_CSV, index=False)

print("✅ Segmentation evaluation saved to:")
print(OUT_CSV)
print("\nMean metrics:")
print(df_metrics.mean(numeric_only=True))


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/glaucoma detection/segmentation_metrics_refuge.csv"
)

print(df.head())
print("\nMissing values per column:\n", df.isna().sum())
print("\nMetric ranges:\n", df.describe())


In [ ]:
print("Pred disc sum:", disc_bin.sum())
print("GT disc sum:", gt_disc.sum())

print("Pred cup sum:", cup_bin.sum())
print("GT cup sum:", gt_cup.sum())


In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("GT Disc")
plt.imshow(gt_disc, cmap="gray")
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Pred Disc")
plt.imshow(disc_bin, cmap="gray")
plt.axis("off")

plt.subplot(1,3,3)
plt.title("GT vs Pred Overlay")
plt.imshow(gt_disc, cmap="gray")
plt.imshow(disc_bin, cmap="jet", alpha=0.4)
plt.axis("off")

plt.show()


In [ ]:
np.unique(gt_disc), np.unique(gt_cup)


In [ ]:
import cv2
import numpy as np
import os

MASK_DIR = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"

sample = sorted(os.listdir(MASK_DIR))[0]
mask = cv2.imread(os.path.join(MASK_DIR, sample), 0)

print("Unique mask values:", np.unique(mask))


In [ ]:
gt_disc = (mask == 1).astype(np.uint8)
gt_cup  = (mask == 2).astype(np.uint8)


In [ ]:
IMAGE_DIR = "/content/drive/MyDrive/glaucoma detection/Images_refuge"
MASK_DIR  = "/content/drive/MyDrive/glaucoma detection/Masks_refuge"
MODEL_PATH = "/content/drive/MyDrive/glaucoma detection/attention_unet_refuge.keras"


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model

model = load_model(MODEL_PATH, compile=False)
print("✅ Model loaded")


In [ ]:
import numpy as np

def dice(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    return (2 * inter) / (np.sum(y_true) + np.sum(y_pred) + 1e-6)

def iou(y_true, y_pred):
    inter = np.sum(y_true & y_pred)
    union = np.sum(y_true | y_pred)
    return inter / (union + 1e-6)

def pixel_acc(y_true, y_pred):
    return np.mean(y_true == y_pred)

def sensitivity(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn + 1e-6)

def specificity(y_true, y_pred):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tn / (tn + fp + 1e-6)


In [ ]:
import cv2
import pandas as pd
import os

rows = []

image_files = sorted(os.listdir(IMAGE_DIR))

for fname in image_files:
    img = cv2.imread(os.path.join(IMAGE_DIR, fname), 0)
    img = cv2.resize(img, (256,256))
    img = img / 255.0
    img = img[..., np.newaxis]
    img = np.expand_dims(img, 0)

    mask = cv2.imread(os.path.join(MASK_DIR, fname.replace(".jpg", ".png")), 0)
    mask = cv2.resize(mask, (256,256), interpolation=cv2.INTER_NEAREST)

    # ✅ CORRECT GT decoding
    gt_disc = (mask == 1).astype(np.uint8)
    gt_cup  = (mask == 2).astype(np.uint8)

    pred_disc, pred_cup = model.predict(img, verbose=0)

    pred_disc = (pred_disc[0,:,:,0] > 0.5).astype(np.uint8)
    pred_cup  = (pred_cup[0,:,:,0]  > 0.5).astype(np.uint8)

    rows.append({
        "image": fname,

        "disc_dice": dice(gt_disc, pred_disc),
        "disc_iou": iou(gt_disc, pred_disc),
        "disc_pixel_acc": pixel_acc(gt_disc, pred_disc),
        "disc_sensitivity": sensitivity(gt_disc, pred_disc),
        "disc_specificity": specificity(gt_disc, pred_disc),

        "cup_dice": dice(gt_cup, pred_cup),
        "cup_iou": iou(gt_cup, pred_cup),
        "cup_pixel_acc": pixel_acc(gt_cup, pred_cup),
        "cup_sensitivity": sensitivity(gt_cup, pred_cup),
        "cup_specificity": specificity(gt_cup, pred_cup),
    })


In [ ]:
df = pd.DataFrame(rows)

SAVE_PATH = "/content/drive/MyDrive/glaucoma detection/segmentation_metrics_refuge_FIXED.csv"
df.to_csv(SAVE_PATH, index=False)

print("✅ Metrics saved to:", SAVE_PATH)


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/glaucoma detection/segmentation_metrics_refuge_FIXED.csv"
)

print(df.describe())
